# RNN Model on IMDB Dataset

The IMDB dataset consists of 50,000 movie reviews, evenly divided into training and testing sets. Each review is encoded as a sequence of integers (tokenized sequences), where each integer represents a specific word. Lower integers correspond to the most frequent words in the dataset.
There are two classes:
- 1 (Positive): Positive movie review
- 0 (Negative): Negative movie review

## Techniques Used:
### Tokenization:
Text reviews are converted into numerical sequences. This notebook retains the top 10,000 most frequently occurring words (`MAX_WORDS=10000`).
For example: *"This movie is great!"* might be represented as `[1, 14, 25, 50]`.

### Padding:
Reviews vary in length, so `pad_sequences()` standardizes the length of all reviews. We set `MAX_LEN=500`, truncating longer sequences and padding shorter ones with 0s.

### Objective:
This dataset is widely used for developing NLP models (sentiment analysis). Preprocessing the text enables RNN models to process sequential data efficiently.

In [ ]:
# Import necessary libraries for data loading and preprocessing
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# Define global variables for the dataset
MAX_WORDS = 10000
MAX_LEN = 500

# Load the IMDB dataset, keeping only the top MAX_WORDS most frequent words
print("Loading data...")
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=MAX_WORDS)

# Pad sequences to ensure uniform length
print(f"Padding sequences to length: {MAX_LEN}...")
x_train = pad_sequences(x_train, maxlen=MAX_LEN)
x_test = pad_sequences(x_test, maxlen=MAX_LEN)

print(f"Sample review (tokenized and padded):\n{x_train[0]}")
print(f"Label: {y_train[0]}")

## RNN Model with Random Search Memory
Using `KerasTuner` to perform Random Search for optimal hyperparameter selection (**embedding dimension** and **LSTM units**).

In [ ]:
import tensorflow as tf
import keras_tuner as kt

# Define the RNN model-building function
def build_rnn_model(hp):
    model = tf.keras.models.Sequential()
    
    # MAX_WORDS is defined previously, here we explicitly use the value locally
    model.add(tf.keras.layers.Embedding(
        input_dim=MAX_WORDS, 
        output_dim=hp.Int('embedding_dim', min_value=32, max_value=128, step=32)
    ))
    
    model.add(tf.keras.layers.LSTM(
        units=hp.Int('units', min_value=32, max_value=128, step=32)
    ))
    
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Initialize Random Search using KerasTuner
tuner_rnn_random = kt.RandomSearch(
    build_rnn_model,
    objective='val_accuracy',
    max_trials=5, 
    directory='../data/random_search_dir', 
    project_name='imdb_rnn_random',
    overwrite=True
)

# Start searching for the best hyperparameters
print("Starting Random Search...")
tuner_rnn_random.search(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

# Display the optimal hyperparameters found
best_hps_random = tuner_rnn_random.get_best_hyperparameters(num_trials=1)[0]
print(f"Best embedding dimension (Random Search): {best_hps_random.get('embedding_dim')}")
print(f"Best number of LSTM units (Random Search): {best_hps_random.get('units')}")

# Build and train the most optimal Random Search model
best_rnn_model_random = tuner_rnn_random.hypermodel.build(best_hps_random)
history_random = best_rnn_model_random.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

## Model Evaluation (Random Search)
Evaluate the tuned model using performance metrics such as **Accuracy, Precision, Recall, and F1-score.**

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

# Predict the outcomes on the test set
# For probabilities > 0.5, we assign label 1 (Positive), else 0 (Negative)
y_pred_random = (best_rnn_model_random.predict(x_test) > 0.5).astype('int32')

# Generate the confusion matrix
cm_random = confusion_matrix(y_test, y_pred_random)

# Plot the Confusion Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm_random, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Random Search Optimal Model')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

# Calculate performance metrics
accuracy_random = accuracy_score(y_test, y_pred_random)
precision_random = precision_score(y_test, y_pred_random, average='binary')
recall_random = recall_score(y_test, y_pred_random, average='binary')
f1_random = f1_score(y_test, y_pred_random, average='binary')

# Display performance metrics
print(f"Evaluation Metrics (Random Search):")
print(f"Accuracy:  {accuracy_random:.4f}")
print(f"Precision: {precision_random:.4f}")
print(f"Recall:    {recall_random:.4f}")
print(f"F1-score:  {f1_random:.4f}")

## RNN Model with Grid Search
Utilizing **Grid Search** to exhaustively test a designated set of hyperparameters.

In [ ]:
# Initialize Grid Search using KerasTuner
tuner_rnn_grid = kt.GridSearch(
    build_rnn_model,
    objective='val_accuracy',
    max_trials=5, # Number of distinct trials
    directory='../data/grid_search_dir', 
    project_name='imdb_rnn_grid',
    overwrite=True
)

# Start searching
print("Starting Grid Search...")
tuner_rnn_grid.search(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

# Retrieve the best hyperparameters
best_hps_grid = tuner_rnn_grid.get_best_hyperparameters(num_trials=1)[0]
print(f"Best embedding dimension (Grid Search): {best_hps_grid.get('embedding_dim')}")
print(f"Best number of LSTM units (Grid Search): {best_hps_grid.get('units')}")

# Build and train the most optimal Grid Search model
best_rnn_model_grid = tuner_rnn_grid.hypermodel.build(best_hps_grid)
history_grid = best_rnn_model_grid.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

## Model Evaluation (Grid Search)
Evaluate the performance identical to the Random Search model.

In [ ]:
# Predict outcomes using the optimal Grid Search model
y_pred_grid = (best_rnn_model_grid.predict(x_test) > 0.5).astype('int32')

# Generate Confusion Matrix
cm_grid = confusion_matrix(y_test, y_pred_grid)

# Display the Confusion Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm_grid, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Grid Search Optimal Model')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

# Calculate performance metrics
accuracy_grid = accuracy_score(y_test, y_pred_grid)
precision_grid = precision_score(y_test, y_pred_grid, average='binary')
recall_grid = recall_score(y_test, y_pred_grid, average='binary')
f1_grid = f1_score(y_test, y_pred_grid, average='binary')

# Display performance metrics
print(f"Evaluation Metrics (Grid Search):")
print(f"Accuracy:  {accuracy_grid:.4f}")
print(f"Precision: {precision_grid:.4f}")
print(f"Recall:    {recall_grid:.4f}")
print(f"F1-score:  {f1_grid:.4f}")